# Structured Output (Basic)

| **Concept** | **Purpose** |
|---|---|
| **Free-text output** | Normal LLM reply (just a string), hard to use in code |
| **Pydantic BaseModel** | Define the shape (fields + types) we want |
| **Field(description=...)** | Tell the LLM what each field means |
| **with_structured_output()** | Make the LLM return data in that shape |
| **List of models** | Get many items (like 3 movies) in one response |


> **Key Idea:** LLM normally gives **text**. Structured output makes it give **data** (name, email, age as proper fields) so we can use it directly in code.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_google_genai import ChatGoogleGenerativeAI
# llm = ChatGoogleGenerativeAI(model="gemini-3.8-flash")

In [31]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="qwen2.5:1.5b",
    temperature=0
)

### 1. The Problem

**Short note:** We have messy text and want name, email, age out of it. The LLM replies with a normal sentence (a string). We can't safely do res.name or res.age on a string

In [8]:
text = "Hello, My name is Mahesh  " \
"Email is mahesh@example.com and my age is 21"

In [9]:
res = llm.invoke(f"please give me only name, email and age from this text:{text}")

In [12]:
print(res.content[0]["text"])
print(type(res.content))   # str, just text
     

Name: Mahesh
Email: mahesh@example.com
Age: 21
<class 'list'>


### 2. Pydantic BaseModel + Field

**Short note:** Pydantic lets us define **what the output should look like,** like a form with fields.

--> BaseModel → the class that holds the fields

--> name: str → field name and its type

--> Field(description=...) → tells the LLM what to put in that field

In [20]:
from pydantic import BaseModel, Field

class ResponseStructure(BaseModel):
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    age: int = Field(description="The age of the person")



### 3. with_structured_output()

**Short note:** Give the model our class. It now returns an **object of that class** instead of text.

In [19]:
structure_llm = llm.with_structured_output(ResponseStructure)

In [21]:
res = structure_llm.invoke(f"please give me only name, email and age from this text:{text}")
print(res)
print(type(res))

name='Mahesh' email='mahesh@example.com' age=21
<class '__main__.ResponseStructure'>


In [22]:
# now we can use it like normal Python data
print(res.name)
print(res.email)
print(res.age + 1)   # age is a real int, so maths works

Mahesh
mahesh@example.com
22


In [25]:
# convert to dictionary / JSON if needed
print(res.model_dump())

{'name': 'Mahesh', 'email': 'mahesh@example.com', 'age': 21}


### 4. One Item vs Many Items

**Short note:** A model describes **one** item. If we want many (like 3 movies), wrap it in a List.

Common mistake: movies: Movies gives only one. Use movies: List[Movies] for many.


In [32]:
from typing import List
class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The release year of the movie")

class AllMovies(BaseModel):
    movies: List[Movie] 



movie_llm = llm.with_structured_output(AllMovies)

In [42]:
res = movie_llm.invoke("Give me 3 recent Hindi movies")


In [43]:
res 

AllMovies(movies=[Movie(title='Bhoomi', year=2023), Movie(title='Dilwale Dulhania Le Jayenge', year=2016), Movie(title='Dangal', year=2019)])

## Quick Summary

| **Concept** | **One Line** | **Code Idea** |
|---|---|---|
| **Free-text output** | LLM gives a plain string | `res.content` |
| **Pydantic BaseModel** | Define the shape we want | `class UserInfo(BaseModel)` |
| **Field** | Explain each field to the LLM | `Field(description="...")` |
| **with_structured_output** | LLM returns our class | `llm.with_structured_output(UserInfo)` |
| **List of models** | Many items at once | `movies: List[Movie]` |

### Common Mistakes

- Package name is `pydantic` (not `pydentic`)
- Use `Field(...)`, not some other class
- Method is `invoke` (not `inovoke`)
- Use `List[Movie]` when you want many items